# 🚗 Seatbelt Detection — Fine-tuning a Hugging Face Vision Transformer (**CPU version, saves to Google Drive**)

**Use case:** given an image of a driver/front-seat occupant (e.g. a windshield-view frame from a traffic camera), decide **`seatbelt`** vs **`no_seatbelt`** so violations can be flagged automatically (e-challan, fleet safety, insurance).

### How the system works
```
Camera frame ──► (optional) YOLOv8 finds the person ──► crop ──►
   fine-tuned ViT classifier ──► P(no_seatbelt) ──► compare with tuned threshold ──► ✅ belted / 🚨 violation
```

### What changed for CPU
* Runs on a normal **CPU** Colab runtime — no GPU needed. (If a GPU happens to be available it is used automatically.)
* Model is **`facebook/deit-small-patch16-224`** — the same Vision-Transformer architecture as ViT-Base, ImageNet-pretrained, but ~4× fewer computations, so training finishes in well under an hour on CPU instead of many hours.
* **Everything important is saved to Google Drive** in `MyDrive/Seatbelt_Detection/`: training checkpoints (so a disconnect doesn't lose your progress — just re-run all cells and it **resumes**), the final model, and `seatbelt_model.zip` for the local app.
* The final model is re-loaded from Drive and checked to give **exactly the same predictions** as the evaluated model, so the local app shows the same results you report.

### Steps
| Step | What happens |
|---|---|
| 1 | Install libraries, connect Google Drive |
| 2 | Download the Kaggle dataset `lavdeep1234/driver-seat-belt-dectection` |
| 3 | Auto-detect its format (class folders / YOLO / Pascal-VOC) and build a clean 2-class image set |
| 4 | Remove near-duplicate leakage and make a **stratified, group-aware** train/val/test split |
| 5 | Augmentation + preprocessing |
| 6 | Load pretrained DeiT-Small (ViT) from the Hugging Face Hub |
| 7 | Fine-tune on CPU with a **class-weighted loss**, checkpoints on Drive |
| 8 | Tune the decision threshold on validation, then evaluate **once** on the untouched test set |
| 9 | Look at the mistakes |
| 10 | Save the final model + zip to Google Drive (optional: push to Hugging Face Hub) |
| 11 | Re-load from Drive and verify it reproduces the test results |
| 12 | Download the zip → run the **local upload UI** (`seatbelt_app`) on your laptop |

> ⚠️ **About "100% accuracy".** No real-world model is 100% accurate, and a notebook that *reports* 100% on this dataset is usually measuring leakage (near-identical video frames in train and test) or a tiny negative class. This notebook is built to give you an **honest** number. Report **recall on `no_seatbelt`** and **macro-F1**, not just accuracy: with ~15 belted images per unbelted one, a model that always says "seatbelt" already gets ~93% accuracy while catching zero violators.

## Step 1 — Setup (CPU runtime) and Google Drive
Runtime ▸ Change runtime type ▸ **CPU** is fine. Run the cells below. When the Drive cell runs, a pop-up asks you to allow access — choose your Google account and click **Allow**.

In [ ]:
!pip -q install transformers datasets accelerate kagglehub imagehash scikit-learn
import os, sys, torch, transformers, datasets
torch.set_num_threads(os.cpu_count())          # use every CPU core
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| transformers", transformers.__version__, "| datasets", datasets.__version__)
print("Device:", DEVICE, "| CPU cores:", os.cpu_count(),
      "| (GPU found - it will be used automatically)" if DEVICE == "cuda" else "| running on CPU ✔")

In [ ]:
# ---- Connect Google Drive: everything important is saved here ----------------
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = "/content/drive/MyDrive/Seatbelt_Detection"
else:                                            # running outside Colab (e.g. Jupyter on a laptop)
    DRIVE_DIR = os.path.abspath("Seatbelt_Detection")
os.makedirs(DRIVE_DIR, exist_ok=True)
print("Saving to:", DRIVE_DIR)

In [ ]:
# ---- Configuration (change things here) -------------------------------------
KAGGLE_DATASET = "lavdeep1234/driver-seat-belt-dectection"
MODEL_ID       = "facebook/deit-small-patch16-224"   # CPU-friendly ViT. Even faster: "facebook/deit-tiny-patch16-224". Original (slow on CPU): "google/vit-base-patch16-224"
HUB_REPO       = "your-hf-username/vit-seatbelt-detector"   # only used if you push to the Hub in Step 10b
WORK_DIR       = "/content/seatbelt_cls" if IN_COLAB else os.path.abspath("seatbelt_cls")   # cleaned images (temporary, rebuilt every run)
CKPT_DIR       = f"{DRIVE_DIR}/checkpoints"         # training checkpoints on Drive -> training resumes after a disconnect
SAVE_DIR       = f"{DRIVE_DIR}/seatbelt_model"      # final model on Drive (used by the local app)
CLASSES        = ["no_seatbelt", "seatbelt"]      # index 0 = violation (the class we care about most)
SEED           = 42
EPOCHS         = 15                               # early stopping usually ends training sooner
LR             = 5e-5
BATCH          = 16                               # 16 is safe for CPU RAM
# If the auto-labeller in Step 3 can't understand a class name, map it by hand here, e.g. {"0": "seatbelt", "1": "no_seatbelt", "2": None}
MANUAL_MAP     = {}
# ------------------------------------------------------------------------------
import random, numpy as np
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

## Step 2 — Download the dataset from Kaggle
`kagglehub` downloads public datasets directly. If it asks for credentials: Kaggle ▸ Settings ▸ **Create New API Token**, then run `kagglehub.login()` and paste your username + key.

*Fallback:* if you want more `no_seatbelt` examples, any Roboflow Universe seatbelt dataset exported as **YOLOv8** or **Folder Structure** also works with Step 3 — just set `DATA_ROOT` to its unzipped folder.

In [ ]:
import kagglehub
DATA_ROOT = kagglehub.dataset_download(KAGGLE_DATASET)
print("Downloaded to:", DATA_ROOT)
!find "$DATA_ROOT" -maxdepth 3 -type d | head -40

## Step 3 — Inspect the format and build a clean 2-class image set
Seatbelt datasets come in three shapes, and this cell handles all of them automatically:
* **Class folders** (`seatbelt/…jpg`, `no_seatbelt/…jpg`) → label = folder name
* **YOLO** (`data.yaml` + `images/` + `labels/*.txt`) → every seatbelt/no-seatbelt **box is cropped** (with padding) into its own image
* **Pascal-VOC** (`*.xml` next to images) → same, boxes are cropped

Class names are mapped by keyword (`no`, `un`, `without`, `not worn` → `no_seatbelt`; anything else containing `belt` → `seatbelt`; "not clear"/windshield/vehicle are ignored). **Check the printed mapping and counts** — if anything is wrong, fix it with `MANUAL_MAP` in the config cell and re-run.

In [ ]:
import os, re, glob, shutil, yaml, xml.etree.ElementTree as ET
from pathlib import Path
from collections import Counter
from PIL import Image, ImageOps

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def map_label(raw):
    raw = str(raw)
    if raw in MANUAL_MAP: return MANUAL_MAP[raw]
    n = re.sub(r"[^a-z]", "", raw.lower())
    if not n or "belt" not in n:                        return None   # windshield, vehicle, plate, numeric ids…
    if "notclear" in n or "unclear" in n:              return None
    if (n.startswith(("no", "un", "without", "not")) or "noseat" in n or "nobelt" in n
            or "without" in n or "notworn" in n or "notfastened" in n or "unfastened" in n):
        return "no_seatbelt"
    if n[0] in "dp" and n[1:].startswith("no"):        return "no_seatbelt"  # e.g. DNo-SeatBelt / PNo-SeatBelt
    return "seatbelt"

all_imgs = sorted(p for p in Path(DATA_ROOT).rglob("*") if p.suffix.lower() in IMG_EXT)   # sorted -> same split every run (needed to resume training)
yamls    = sorted(p for p in Path(DATA_ROOT).rglob("*.yaml") if "names" in (yaml.safe_load(open(p)) or {}))
xmls     = sorted(Path(DATA_ROOT).rglob("*.xml"))
print(f"{len(all_imgs)} images | {len(yamls)} YOLO yaml | {len(xmls)} VOC xml")

shutil.rmtree(WORK_DIR, ignore_errors=True)
for c in CLASSES: os.makedirs(f"{WORK_DIR}/{c}", exist_ok=True)
records = []      # (saved_path, label, source_image)  – source is used for leakage-safe splitting

def save_crop(img, box, label, src, k, pad=0.15, min_side=24):
    W, H = img.size
    x1, y1, x2, y2 = box
    bw, bh = x2 - x1, y2 - y1
    x1, y1 = max(0, x1 - pad*bw), max(0, y1 - pad*bh)
    x2, y2 = min(W, x2 + pad*bw), min(H, y2 + pad*bh)
    if min(x2 - x1, y2 - y1) < min_side: return
    out = f"{WORK_DIR}/{label}/{Path(src).stem}_{k}.jpg"
    img.crop((x1, y1, x2, y2)).convert("RGB").save(out, quality=95)
    records.append((out, label, str(src)))

if yamls:                                   # ---------- YOLO format
    names = yaml.safe_load(open(yamls[0]))["names"]
    names = names if isinstance(names, dict) else dict(enumerate(names))
    cls_map = {int(i): map_label(n) for i, n in names.items()}
    print("FORMAT: YOLO   class mapping:", {names[i]: v for i, v in cls_map.items()})
    for ip in all_imgs:
        lp = Path(str(ip).replace(f"{os.sep}images{os.sep}", f"{os.sep}labels{os.sep}")).with_suffix(".txt")
        if not lp.exists(): continue
        img = ImageOps.exif_transpose(Image.open(ip)); W, H = img.size
        for k, line in enumerate(open(lp).read().strip().splitlines()):
            v = line.split()
            if len(v) < 5 or cls_map.get(int(float(v[0]))) is None: continue
            nums = list(map(float, v[1:]))
            if len(nums) == 4:                                   # cx cy w h
                cx, cy, w, h = nums; box = ((cx-w/2)*W, (cy-h/2)*H, (cx+w/2)*W, (cy+h/2)*H)
            else:                                                # polygon x1 y1 x2 y2 …
                xs, ys = nums[0::2], nums[1::2]; box = (min(xs)*W, min(ys)*H, max(xs)*W, max(ys)*H)
            save_crop(img, box, cls_map[int(float(v[0]))], ip, k)

elif xmls:                                  # ---------- Pascal-VOC format
    print("FORMAT: Pascal-VOC")
    by_stem = {p.stem: p for p in all_imgs}
    seen = Counter()
    for xp in xmls:
        root = ET.parse(xp).getroot()
        ip = by_stem.get(xp.stem) or by_stem.get(Path(root.findtext("filename", "")).stem)
        if ip is None: continue
        img = ImageOps.exif_transpose(Image.open(ip))
        for k, obj in enumerate(root.iter("object")):
            name = obj.findtext("name"); lab = map_label(name); seen[(name, lab)] += 1
            if lab is None: continue
            b = obj.find("bndbox")
            box = tuple(float(b.findtext(t)) for t in ("xmin", "ymin", "xmax", "ymax"))
            save_crop(img, box, lab, ip, k)
    print("class mapping:", dict(seen))

else:                                       # ---------- class-folder format
    print("FORMAT: class folders")
    seen = Counter()
    for ip in all_imgs:
        lab = None
        for parent in ip.relative_to(DATA_ROOT).parents:
            if parent.name and (lab := map_label(parent.name)): seen[(parent.name, lab)] += 1; break
        if lab is None: continue
        out = f"{WORK_DIR}/{lab}/{len(records):06d}_{ip.stem}.jpg"
        ImageOps.exif_transpose(Image.open(ip)).convert("RGB").save(out, quality=95)
        records.append((out, lab, str(ip)))
    print("folder → label mapping:", dict(seen))

counts = Counter(r[1] for r in records)
print("\nFINAL CLASS COUNTS:", dict(counts))
assert all(counts[c] > 0 for c in CLASSES), "One class is empty — check the mapping above and set MANUAL_MAP."

In [ ]:
# Look at a few samples of each class — always eyeball your data before training
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 6, figsize=(16, 6))
for row, c in enumerate(CLASSES):
    sample = random.sample([r for r in records if r[1] == c], min(6, counts[c]))
    for ax, r in zip(axes[row], sample):
        ax.imshow(Image.open(r[0])); ax.set_title(c); ax.axis("off")
plt.tight_layout(); plt.show()

## Step 4 — Leakage-safe, stratified split (70 / 15 / 15)
Many seatbelt datasets are **frames cut from videos**, so the "same" image appears many times with tiny changes. If one copy lands in train and another in test, the test score is fake (this is where most "100% accuracy" claims come from).

So we compute a **perceptual hash** of every image, put near-identical images (and crops cut from the same photo) into one *group* — the duplicate test tightens itself automatically if it starts merging genuinely different drivers — and split by group with `StratifiedGroupKFold` — near-duplicates always stay on the same side, and both splits keep the class ratio.

In [ ]:
import imagehash
from sklearn.model_selection import StratifiedGroupKFold

# hash the actual image the model will see (the crop), so different drivers from the same fixed camera stay separate
hashes = np.array([int(str(imagehash.phash(Image.open(r[0]).convert("RGB"))), 16) for r in records], dtype=np.uint64)

def popcount(a):
    if hasattr(np, "bitwise_count"): return np.bitwise_count(a)            # numpy >= 2.0
    return np.array([bin(int(x)).count("1") for x in a], dtype=np.int64)

def build_groups(max_hamming):
    parent = list(range(len(records)))
    def find(i):
        while parent[i] != i: parent[i] = parent[parent[i]]; i = parent[i]
        return i
    def union(i, j): parent[find(j)] = find(i)
    first_of_src = {}
    for i, r in enumerate(records):                  # crops cut from the same photo share a group
        if r[2] in first_of_src: union(first_of_src[r[2]], i)
        else: first_of_src[r[2]] = i
    for i in range(len(records) - 1):                # near-identical crops (video frames) share a group
        for j in np.where(popcount(hashes[i+1:] ^ hashes[i]) <= max_hamming)[0] + i + 1:
            union(i, int(j))
    return np.array([find(i) for i in range(len(records))])

# start with a tolerant duplicate test and tighten it if it starts merging genuinely different images
for MAX_HAMMING in (6, 4, 2, 0):
    groups = build_groups(MAX_HAMMING)
    biggest = Counter(groups).most_common(1)[0][1]
    if biggest <= 0.05 * len(records): break
print(f"MAX_HAMMING={MAX_HAMMING}: {len(records)} images -> {len(set(groups))} groups (largest group: {biggest} images)")

paths  = np.array([r[0] for r in records])
labels = np.array([CLASSES.index(r[1]) for r in records])

def split(idx, n_splits):
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    a, b = next(sgkf.split(paths[idx], labels[idx], groups[idx]))
    return idx[a], idx[b]

all_idx = np.arange(len(paths))
trainval_idx, test_idx = split(all_idx, 7)        # ~15% test
train_idx, val_idx     = split(trainval_idx, 6)   # ~15% of total for val
assert not (set(groups[train_idx]) & set(groups[test_idx])) and not (set(groups[train_idx]) & set(groups[val_idx]))

for name, idx in [("train", train_idx), ("val", val_idx), ("test", test_idx)]:
    print(f"{name:5s}: {len(idx):5d} images |", {c: int((labels[idx] == i).sum()) for i, c in enumerate(CLASSES)})

# keep a record of the split on Drive (useful for your report)
import json
with open(f"{DRIVE_DIR}/split.json", "w") as f:
    json.dump({n: [str(p) for p in paths[idx]] for n, idx in [("train", train_idx), ("val", val_idx), ("test", test_idx)]}, f)

In [ ]:
from datasets import Dataset, DatasetDict, Features, ClassLabel, Image as HFImage

features = Features({"image": HFImage(), "label": ClassLabel(names=CLASSES)})
make = lambda idx: Dataset.from_dict({"image": paths[idx].tolist(), "label": labels[idx].tolist()}, features=features)
ds = DatasetDict(train=make(train_idx), validation=make(val_idx), test=make(test_idx))
ds

## Step 5 — Preprocessing and augmentation
The pretrained model tells us the input size (224×224) and the normalisation. Training images get random crops, flips (a mirrored belt is still a belt, and it covers both left- and right-hand-drive cars), small rotations, colour jitter (day/night/glare) and blur (motion, low-res cameras). Validation/test images are only resized — **and the local app uses exactly this same resize + normalise**, so its predictions match the numbers you get here.

In [ ]:
from transformers import AutoImageProcessor
from torchvision import transforms as T

processor = AutoImageProcessor.from_pretrained(MODEL_ID)
from transformers import AutoConfig
size = AutoConfig.from_pretrained(MODEL_ID).image_size   # 224 - the size the model's position embeddings were trained for
print("input size:", size, "| mean:", processor.image_mean, "| std:", processor.image_std)
norm = T.Normalize(mean=processor.image_mean, std=processor.image_std)

train_tf = T.Compose([
    T.RandomResizedCrop(size, scale=(0.75, 1.0), ratio=(0.8, 1.25)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(10),
    T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.03),
    T.RandomApply([T.GaussianBlur(5, sigma=(0.1, 2.0))], p=0.2),
    T.RandomGrayscale(p=0.1),          # many traffic cameras are IR / monochrome at night
    T.ToTensor(), norm,
])
eval_tf = T.Compose([T.Resize((size, size)), T.ToTensor(), norm])

def apply(tf):
    def _f(batch):
        batch["pixel_values"] = [tf(img.convert("RGB")) for img in batch["image"]]
        return batch
    return _f

ds["train"].set_transform(apply(train_tf))
ds["validation"].set_transform(apply(eval_tf))
ds["test"].set_transform(apply(eval_tf))

def collate_fn(batch):
    return {"pixel_values": torch.stack([b["pixel_values"] for b in batch]),
            "labels": torch.tensor([b["label"] for b in batch])}

## Step 6 — Load the pretrained model from the Hugging Face Hub
`facebook/deit-small-patch16-224` is a Vision Transformer (same architecture as `google/vit-base`, just smaller) trained on ImageNet. We keep its learned visual features and replace its 1000-class head with a new 2-class head (`ignore_mismatched_sizes=True`) — that is what **fine-tuning** means.

In [ ]:
from transformers import AutoModelForImageClassification

id2label = dict(enumerate(CLASSES)); label2id = {v: k for k, v in id2label.items()}
model = AutoModelForImageClassification.from_pretrained(
    MODEL_ID, num_labels=len(CLASSES), id2label=id2label, label2id=label2id,
    ignore_mismatched_sizes=True,
)
print(f"{sum(p.numel() for p in model.parameters())/1e6:.1f}M parameters")

## Step 7 — Fine-tune with a class-weighted loss (on CPU, checkpoints on Drive)
Because belted images vastly outnumber unbelted ones, a plain loss would learn to say "seatbelt" for everything. We weight each class by the inverse of its frequency, so one missed violator costs as much as many missed belted drivers. The best epoch is chosen by **macro-F1 on validation** (not accuracy), and training stops early if validation stops improving.

⏱️ On a CPU runtime expect roughly **2–5 minutes per epoch** (the progress bar shows the remaining time). A checkpoint is written to Drive after every epoch. **If Colab disconnects:** reconnect, *Runtime ▸ Run all*, and training continues from the last saved epoch.

ℹ️ With the newest `transformers` you may see a yellow log line *"There were missing keys in the checkpoint model loaded…"*. It is harmless here: the cells below re-load the weights with `from_pretrained`, which reads them correctly (this was tested).

In [ ]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

cls_counts = np.bincount(labels[train_idx], minlength=len(CLASSES))
class_weights = torch.tensor(len(train_idx) / (len(CLASSES) * cls_counts), dtype=torch.float)
print("class weights:", {c: round(float(w), 2) for c, w in zip(CLASSES, class_weights)})

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels_ = inputs.pop("labels")
        outputs = model(**inputs)
        loss = torch.nn.functional.cross_entropy(outputs.logits, labels_, weight=class_weights.to(outputs.logits.device))
        return (loss, outputs) if return_outputs else loss

NEG = CLASSES.index("no_seatbelt")
def compute_metrics(eval_pred):
    logits, y = eval_pred
    p = logits.argmax(-1)
    return {
        "accuracy": accuracy_score(y, p),
        "f1_macro": f1_score(y, p, average="macro"),
        "no_seatbelt_recall":    recall_score(y, p, pos_label=NEG, zero_division=0),
        "no_seatbelt_precision": precision_score(y, p, pos_label=NEG, zero_division=0),
    }

import math
warmup_steps = math.ceil(0.1 * EPOCHS * math.ceil(len(train_idx) / BATCH))   # 10% warm-up
args = TrainingArguments(
    output_dir=CKPT_DIR,                     # on Google Drive
    eval_strategy="epoch", save_strategy="epoch",
    learning_rate=LR, per_device_train_batch_size=BATCH, per_device_eval_batch_size=BATCH,
    num_train_epochs=EPOCHS, warmup_steps=warmup_steps, weight_decay=0.01,
    load_best_model_at_end=True, metric_for_best_model="f1_macro", greater_is_better=True,
    save_total_limit=2, logging_steps=10, fp16=torch.cuda.is_available(),
    remove_unused_columns=False, seed=SEED, report_to="none",
    dataloader_num_workers=2 if DEVICE == "cuda" else 0,   # on CPU, extra loader processes just compete with training for the same cores
)
trainer = WeightedTrainer(
    model=model, args=args, data_collator=collate_fn,
    train_dataset=ds["train"], eval_dataset=ds["validation"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=4)],
)

In [ ]:
# resume automatically if an earlier run already saved checkpoints on Drive
from transformers.trainer_utils import get_last_checkpoint
last_ckpt = get_last_checkpoint(CKPT_DIR) if os.path.isdir(CKPT_DIR) else None
if last_ckpt:
    print("Resuming from the last checkpoint on Drive:", last_ckpt)
    # load the weights with from_pretrained (handles every transformers version's layer names correctly);
    # the Trainer then restores the optimizer, learning-rate schedule and epoch counter
    trainer.model.load_state_dict(AutoModelForImageClassification.from_pretrained(last_ckpt).state_dict())
else:
    print("Starting training from scratch...")
train_result = trainer.train(resume_from_checkpoint=last_ckpt)
print(train_result.metrics)

# make sure the BEST epoch (by val macro-F1) is the model we evaluate and save
best_ckpt = trainer.state.best_model_checkpoint
print("best checkpoint:", best_ckpt, "| best val macro-F1:", trainer.state.best_metric)
trainer.model = trainer.model_wrapped = AutoModelForImageClassification.from_pretrained(best_ckpt).to(trainer.args.device)

In [ ]:
# Training curves
hist = [h for h in trainer.state.log_history]
tr = [(h["epoch"], h["loss"]) for h in hist if "loss" in h]
ev = [(h["epoch"], h["eval_loss"], h["eval_f1_macro"]) for h in hist if "eval_loss" in h]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(*zip(*tr), label="train loss"); ax[0].plot([e[0] for e in ev], [e[1] for e in ev], "o-", label="val loss"); ax[0].legend(); ax[0].set_xlabel("epoch")
ax[1].plot([e[0] for e in ev], [e[2] for e in ev], "o-"); ax[1].set_title("val macro-F1"); ax[1].set_xlabel("epoch")
plt.show()

## Step 8 — Tune the decision threshold, then test once
By default a model says "no_seatbelt" when that probability > 0.5. For enforcement you usually want to catch more violators, so we pick the threshold on the **validation** set that maximises macro-F1, while keeping `no_seatbelt` recall high. Then the **test set is used exactly once** — that number is the one you report.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score

def probs_no_belt(split):
    out = trainer.predict(ds[split])
    pr = torch.softmax(torch.tensor(out.predictions), -1)[:, NEG].numpy()
    return pr, out.label_ids

val_p, val_y = probs_no_belt("validation")
cands = []
for t in np.linspace(0.05, 0.95, 91):
    pred = np.where(val_p >= t, NEG, 1 - NEG)
    cands.append((f1_score(val_y, pred, average="macro"), recall_score(val_y, pred, pos_label=NEG, zero_division=0), t))
best_f1 = max(c[0] for c in cands)
# among thresholds within 0.01 macro-F1 of the best, take the one with the highest violator recall
THRESH = max([c for c in cands if c[0] >= best_f1 - 0.01], key=lambda c: (c[1], c[0]))[2]
print(f"chosen threshold P(no_seatbelt) >= {THRESH:.2f}")

test_p, test_y = probs_no_belt("test")
test_pred = np.where(test_p >= THRESH, NEG, 1 - NEG)
print(classification_report(test_y, test_pred, target_names=CLASSES, digits=4))
print("ROC-AUC (no_seatbelt):", round(roc_auc_score(test_y == NEG, test_p), 4))
ConfusionMatrixDisplay(confusion_matrix(test_y, test_pred), display_labels=CLASSES).plot(cmap="Blues"); plt.show()

**How to read this:**
* **no_seatbelt recall** = of all real violators, how many we caught. This is the number that matters for enforcement.
* **no_seatbelt precision** = of everyone we flagged, how many really were unbelted (false challans hurt trust).
* If the test set only has a few dozen `no_seatbelt` images, one mistake moves recall by several %. Say so when you present.

## Step 9 — Error analysis
Every wrong prediction on the test set, with the model's confidence. Look for patterns: dark clothing, glare, tinted glass, belt hidden by an arm, wrong labels in the dataset.

In [ ]:
wrong = np.where(test_pred != test_y)[0]
print(f"{len(wrong)} mistakes out of {len(test_y)}")
raw_test = Dataset.from_dict({"image": paths[test_idx].tolist()}, features=Features({"image": HFImage()}))
n = min(12, len(wrong))
if n:
    fig, axes = plt.subplots(1, n, figsize=(2.6*n, 3.2)); axes = np.atleast_1d(axes)
    for ax, i in zip(axes, wrong[:n]):
        ax.imshow(raw_test[int(i)]["image"]); ax.axis("off")
        ax.set_title(f"true {CLASSES[test_y[i]]}\nP(no)={test_p[i]:.2f}", fontsize=8)
    plt.tight_layout(); plt.show()

## Step 10 — Save the final model to Google Drive
**10a** saves the best model to `MyDrive/Seatbelt_Detection/seatbelt_model/` and makes **`seatbelt_model.zip`** next to it — that zip is what the local app uses. It also writes `inference_config.json` (input size, normalisation, classes and the tuned threshold) so the app preprocesses images exactly like the test set.

**10b** *(optional, off by default)* uploads to the Hugging Face Hub. **Token:** huggingface.co ▸ Settings ▸ Access Tokens ▸ *Create new token* ▸ type **Write**. In Colab click the 🔑 key icon ▸ *Add new secret* ▸ Name `HF_TOKEN`, turn **Notebook access** on (or paste it when asked).

In [ ]:
# ---- Step 10a: save model + inference settings + zip to Google Drive ---------
import os, json, shutil
from transformers import ViTImageProcessor
from PIL import Image as PILImage

shutil.rmtree(SAVE_DIR, ignore_errors=True)
trainer.model.config.no_seatbelt_threshold = float(THRESH)
trainer.save_model(SAVE_DIR)

# a processor that does exactly what eval_tf does (resize to size x size, no centre-crop),
# so even Hugging Face's pipeline() sees the same input as our test set
ViTImageProcessor(do_resize=True, size={"height": size, "width": size}, resample=PILImage.BILINEAR,
                  do_rescale=True, do_normalize=True,
                  image_mean=list(processor.image_mean), image_std=list(processor.image_std)).save_pretrained(SAVE_DIR)

inference_cfg = {
    "classes": CLASSES,
    "image_size": int(size),
    "image_mean": [float(x) for x in processor.image_mean],
    "image_std":  [float(x) for x in processor.image_std],
    "no_seatbelt_threshold": float(THRESH),
    "base_model": MODEL_ID,
    "test_metrics": {
        "macro_f1": float(f1_score(test_y, test_pred, average="macro")),
        "accuracy": float(accuracy_score(test_y, test_pred)),
        "no_seatbelt_recall": float(recall_score(test_y, test_pred, pos_label=NEG, zero_division=0)),
        "no_seatbelt_precision": float(precision_score(test_y, test_pred, pos_label=NEG, zero_division=0)),
        "n_test_images": int(len(test_y)),
    },
}
with open(f"{SAVE_DIR}/inference_config.json", "w") as f:
    json.dump(inference_cfg, f, indent=2)

shutil.make_archive(SAVE_DIR, "zip", SAVE_DIR)       # -> .../Seatbelt_Detection/seatbelt_model.zip
print("✅ Model saved on Drive:", SAVE_DIR, "->", sorted(os.listdir(SAVE_DIR)))
print("✅ Zip for the local app:", SAVE_DIR + ".zip", f"({os.path.getsize(SAVE_DIR + '.zip')/1e6:.0f} MB)")
print(json.dumps(inference_cfg["test_metrics"], indent=2))

In [ ]:
# ---- Step 10b: upload to the Hugging Face Hub (optional, never crashes) -------
PUSH_TO_HUB = False       # set to True to also upload to the Hugging Face Hub

def get_hf_token():
    try:
        from google.colab import userdata
        tok = userdata.get("HF_TOKEN")
        if tok:
            print("Using the HF_TOKEN Colab secret.")
            return tok.strip()
    except Exception as e:
        print(f"No usable HF_TOKEN secret ({type(e).__name__}) - paste the token instead.")
    from getpass import getpass
    return getpass("Paste your Hugging Face WRITE token (starts with hf_): ").strip()

HINTS = {
    "401": "Token is wrong or deleted. Create a new WRITE token and try again.",
    "403": "Token has no write permission. Create a WRITE token (or tick 'write to repos' on a fine-grained token).",
    "404": "Repo/user not found. Check that HUB_REPO starts with your own Hugging Face username.",
    "SecretNotFound": "Secret name must be exactly HF_TOKEN.",
    "NotebookAccess": "Turn on 'Notebook access' for the HF_TOKEN secret.",
}

if PUSH_TO_HUB:
    try:
        from huggingface_hub import whoami, create_repo
        HF_TOKEN = get_hf_token()
        if not HF_TOKEN.startswith("hf_"):
            raise ValueError("That is not a Hugging Face token - it must start with hf_")
        me = whoami(token=HF_TOKEN)
        role = me.get("auth", {}).get("accessToken", {}).get("role")
        print("Logged in as:", me["name"], "| token role:", role)
        if role == "read":
            raise PermissionError("403 - this token is READ-only")
        if HUB_REPO.split("/")[0] != me["name"]:
            HUB_REPO = f"{me['name']}/vit-seatbelt-detector"
        create_repo(HUB_REPO, exist_ok=True, token=HF_TOKEN)
        trainer.model.push_to_hub(HUB_REPO, commit_message="Fine-tuned ViT seatbelt classifier", token=HF_TOKEN)
        ViTImageProcessor.from_pretrained(SAVE_DIR).push_to_hub(HUB_REPO, token=HF_TOKEN)   # the eval-matching processor
        print("✅ Uploaded! Model page: https://huggingface.co/" + HUB_REPO)
    except Exception as e:
        msg = f"{type(e).__name__}: {e}"
        print("❌ Upload did not work:", msg[:300])
        for key, hint in HINTS.items():
            if key in msg:
                print("👉 Fix:", hint)
        print(f"Your model is still saved in '{SAVE_DIR}', so Steps 11 and 12 will work. Re-run this cell after fixing the token.")
else:
    print("Upload skipped (PUSH_TO_HUB = False).")

## Step 11 — Re-load the model from Drive and verify it
This loads the model **from the Drive folder** with the same code the local app uses, runs it on every test image, and checks that the probabilities match the evaluation above. If this cell prints ✅, the app on your laptop will give the same answers you reported.

In [ ]:
import json, numpy as np, torch
from PIL import Image, ImageOps
from torchvision import transforms as T
from transformers import AutoModelForImageClassification

# ---- exactly the same loading + preprocessing as seatbelt_app/app.py --------
icfg = json.load(open(f"{SAVE_DIR}/inference_config.json"))
saved_model = AutoModelForImageClassification.from_pretrained(SAVE_DIR).eval()
app_tf = T.Compose([T.Resize((icfg["image_size"], icfg["image_size"])), T.ToTensor(),
                    T.Normalize(icfg["image_mean"], icfg["image_std"])])
NO_IDX = saved_model.config.label2id["no_seatbelt"]
THR = icfg["no_seatbelt_threshold"]

@torch.no_grad()
def p_no_seatbelt(images):
    x = torch.stack([app_tf(ImageOps.exif_transpose(im).convert("RGB")) for im in images])
    return torch.softmax(saved_model(pixel_values=x).logits.float(), -1)[:, NO_IDX].numpy()

def check_seatbelt(img):
    if not isinstance(img, Image.Image):
        img = Image.open(img)
    p_no = float(p_no_seatbelt([img])[0])
    verdict = "🚨 NO SEATBELT — violation" if p_no >= THR else "✅ Seatbelt worn"
    return verdict, {"no_seatbelt": round(p_no, 4), "seatbelt": round(1 - p_no, 4)}

# ---- verify on the whole test set ------------------------------------------
reloaded = np.concatenate([p_no_seatbelt([Image.open(p) for p in paths[test_idx][i:i+32]])
                           for i in range(0, len(test_idx), 32)])
max_diff = float(np.abs(reloaded - test_p).max())
same_decisions = int(((reloaded >= THR) == (test_p >= THR)).sum())
print(f"max probability difference vs. evaluation: {max_diff:.2e}")
print(f"same decision on {same_decisions}/{len(test_p)} test images")
# (on a GPU runtime evaluation ran in fp16, so allow tiny rounding differences right at the threshold)
real_flips = ((reloaded >= THR) != (test_p >= THR)) & (np.abs(test_p - THR) > 1e-2)
assert max_diff < 1e-2 and not real_flips.any(), "Saved model does not reproduce the evaluation!"
print("✅ The model saved on Drive reproduces the test results exactly.")

for cls_i, name in enumerate(CLASSES):
    sample = [p for p, l in zip(paths[test_idx], labels[test_idx]) if l == cls_i][:1]
    if sample:
        print(f"true: {name:12s} ->", check_seatbelt(sample[0]))

## Step 12 — Run the upload UI on your own laptop
1. In **Google Drive** open `My Drive ▸ Seatbelt_Detection` and download **`seatbelt_model.zip`**.
2. Put the zip inside the **`seatbelt_app`** folder (next to `app.py`). Don't unzip it — the app does that itself.
3. Open a terminal in `seatbelt_app` and run:
   * **Windows:** double-click `run_windows.bat`
   * **Mac / Linux:** `bash run_mac_linux.sh`
4. Your browser opens **http://127.0.0.1:7860** → upload a photo → you see ✅ / 🚨, the probabilities, and the tuned threshold.

Two modes in the app:
* **Driver crop** — for images like the training data (a cropped driver / windshield view). This is the most accurate mode.
* **Full scene** — a COCO-pretrained YOLOv8 first finds each person, then every person crop is classified and boxed (green = belted, red = violation).

Everything runs on your CPU; after the first install no internet is needed (except the one-time YOLOv8 download for Full-scene mode).

## Limitations to state honestly (viva / report)
* **Domain shift:** the dataset's camera angle, country and lighting differ from Pakistani roads; expect lower accuracy on local footage until you add local images and re-run this notebook.
* **Small negative class:** few `no_seatbelt` examples → test metrics have wide error bars. Collect more violations first; that improves the model more than any architecture change.
* **Hard cases:** night/IR, heavy glare, tinted glass, dark belt on dark clothes, passenger vs driver.
* **Human in the loop:** for e-challans, a flagged image should be reviewed by a person before a fine is issued.